In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.2 Root-Finding

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.2",
    title="Root-Finding",
    blurb="Solving f(x)=0 when there is no formula: bisection, Newton, and "
    "the secant method — their convergence orders, their failure "
    "modes, and the engine behind the transcendental equations of "
    "quantum mechanics.",
    difficulty="intermediate",
    estimate="80–110 min",
)

## Notebook overview

We learned what a root is in secondary school: a value $x_\star$ with
$f(x_\star)=0$, and how to find one when the algebra cooperates: factor the
quadratic, invert the function. The question then becomes what to do when it
*doesn't*: when $f$ has no closed-form inverse, which is the overwhelmingly
common case in physics. How does one actually find $x_\star$ then, how fast, and
how can it go wrong?

This notebook builds the three classical answers from scratch (**bisection**,
**Newton's method**, and the **secant method**) and measures the property that
separates them: their **order of convergence**, how quickly the error shrinks
per step. We will see bisection crawl (one bit at a time, but never failing),
Newton sprint (doubling its correct digits each step) and then spectacularly
*fail* by cycling forever, and the secant method land in between at a rate set
by the golden ratio. We close on `scipy.optimize.brentq`, the safe-and-fast
hybrid that is the course's default.

This is foundational in the literal sense: the solver we build here is the
**engine** for the transcendental eigenvalue equations of quantum mechanics:
the finite square well in
[§6.11](../06-quantum-mechanics/bound-states-1d.ipynb), and $\tan x = x$ for the
infinite spherical well in
[§6.16](../06-quantum-mechanics/central-potentials-3d.ipynb) (which we solve, as
a preview, in Exercise 7), and for the orbital
turning points of [§2.4](../02-classical-mechanics/central-force.ipynb).
Familiar goal, unfamiliar machinery.

> **How to read the checks.** Each exercise ends with a `validate` call that
> compares our result to an independent truth: a known root, a predicted
> convergence order, an exact special value. A ✓ is strong evidence we got it
> right; a ✗ is not a verdict but a prompt to *locate the discrepancy* (a
> bracket that strayed across an asymptote, an order estimated on the noisy tail
> near machine precision, a derivative typo). Passing is strong evidence, not
> proof.

> **Scope.** A working review, not a numerical-analysis course. For the full
> treatment see Press et al., *Numerical Recipes*, ch. 9 {cite}`numrecipes`, and
> Higham {cite}`higham2002` on the floating-point limits that set the accuracy
> floor ([§0.1](floating-point.ipynb)).

## Theory in brief

### The problem and two families of method

Given continuous $f$, find $x_\star$ with $f(x_\star)=0$. The methods split into
two families. **Bracketing** methods keep an interval known to contain a root
and shrink it: guaranteed to converge, but slowly. **Open** (iterative) methods
generate a sequence $x_n \to x_\star$ from a formula: fast, but with no safety
net: they can diverge, cycle, or land on the wrong root.

### Bisection

If $f(a)\,f(b) < 0$ then $f$ has a root in $[a,b]$ (intermediate value theorem).
Bisection evaluates the midpoint $m=\tfrac12(a+b)$ and keeps whichever half
still brackets the sign change. After $n$ steps the bracket width is

```{math}
:label: eq-bisection
(b-a)_n = \frac{(b-a)_0}{2^{\,n}},
```

so the width halves every step: **linear** convergence, exactly one bit of
accuracy per evaluation, but *guaranteed*. (A subtlety we exploit below: the
*bracket width* halves cleanly, while the *midpoint error* can jump around
inside the shrinking bracket, so convergence is best measured by the width.)

### Newton's method

Newton linearises $f$ at the current guess and steps to where the tangent
crosses zero:

```{math}
:label: eq-newton
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}.
```

Near a *simple* root it converges **quadratically**: the error squares each
step ($e_{n+1}\sim e_n^2$), so the number of correct digits roughly doubles per
iteration (a Taylor expansion of $f$ about the root shows it; Press et al.,
*Numerical Recipes*, §9.4, carry the expansion out in full). The price: it
needs $f'$, it needs a good starting guess, and it
fails outright where $f'=0$ or when the geometry sends it cycling or diverging.

### The secant method

When the derivative is unavailable or expensive, we can approximate it from the
two previous iterates. Replacing $f'(x_n)$ in {eq}`eq-newton` by a finite
difference gives the **secant method**:

```{math}
:label: eq-secant
x_{n+1} = x_n - f(x_n)\,\frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}.
```

Because it needs no derivative, it is often the practical choice; the price for
that convenience is a slightly slower convergence order, the **golden ratio**
$\varphi = \tfrac{1+\sqrt5}{2}\approx 1.618$: still superlinear, sitting neatly
between bisection's linear crawl and Newton's quadratic sprint. (The order
follows from the two-term error recursion $e_{n+1}\sim e_n e_{n-1}$; Press et
al., *Numerical Recipes*, §9.2, sketch the argument.)

### Measuring the order of convergence

The order $p$ is defined by $e_{n+1} \approx C\,e_n^{\,p}$ with
$e_n = |x_n - x_\star|$. Taking logs of three consecutive errors eliminates the
constant $C$ and gives an estimator

```{math}
:label: eq-order
p \approx \frac{\ln(e_{n+1}/e_n)}{\ln(e_n/e_{n-1})}.
```

We must read $p$ from the *mid-range* iterates: the first few have not settled
into the asymptotic regime, and the last few sit on the machine-precision floor
([§0.1](floating-point.ipynb)), where $e_n$ is round-off noise and the ratio is
meaningless.

### Failure modes and the practical default

Newton's failures are vivid: a step through a near-zero $f'$ flings the iterate
far away; some functions and starts make it **cycle** forever; others **diverge**.
Bracketing methods are immune to these but cannot find a root without a sign
change (so they miss even-multiplicity roots). The practical default is
`scipy.optimize.brentq`, **Brent's method**, which combines bisection's guarantee
with the speed of the secant / inverse-quadratic interpolation, and is what we
use throughout the course.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from ecp import draw, validate
from ecp.animate import show

rng = np.random.default_rng(0)
PHI = (1 + np.sqrt(5)) / 2  # golden ratio, the secant method's convergence order


def bisection(f, a, b, tol=1e-13, maxit=200):
    """Bracketing root-find by repeated interval halving.

    Guaranteed but linearly convergent: each step halves the bracket, the safe baseline.

    Parameters
    ----------
    f : callable
        Function with a sign change on ``[a, b]``.
    a, b : float
        Bracket endpoints.
    tol : float, optional
        Bracket-width tolerance.
    maxit : int, optional
        Iteration cap.

    Returns
    -------
    tuple
        The root, the midpoint history, and the bracket-width history.
    """
    fa, fb = f(a), f(b)
    assert fa * fb < 0, "f(a) and f(b) must have opposite signs"
    mids, widths = [], []
    for _ in range(maxit):
        m = 0.5 * (a + b)
        fm = f(m)
        mids.append(m)
        widths.append(b - a)
        if fm == 0.0 or (b - a) < tol:
            break
        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    return m, np.array(mids), np.array(widths)


def newton(f, fprime, x0, tol=1e-14, maxit=100):
    """Newton's method using the analytic derivative.

    Quadratically convergent near a simple root, but can diverge from a poor start.

    Parameters
    ----------
    f : callable
        Function.
    fprime : callable
        Its derivative.
    x0 : float
        Initial guess.
    tol : float, optional
        Convergence tolerance.
    maxit : int, optional
        Iteration cap.

    Returns
    -------
    tuple
        The root and the iterate history.
    """
    x = float(x0)
    hist = [x]
    for _ in range(maxit):
        x = x - f(x) / fprime(x)
        hist.append(x)
        if abs(f(x)) < tol or abs(hist[-1] - hist[-2]) < tol:
            break
    return x, np.array(hist)


def secant(f, x0, x1, tol=1e-14, maxit=100):
    """Secant method using a finite-difference derivative.

    Superlinear — order ≈ 1.618, the golden ratio — with no derivative needed.

    Parameters
    ----------
    f : callable
        Function.
    x0, x1 : float
        Two initial guesses.
    tol : float, optional
        Convergence tolerance.
    maxit : int, optional
        Iteration cap.

    Returns
    -------
    tuple
        The root and the iterate history.
    """
    x0, x1 = float(x0), float(x1)
    f0, f1 = f(x0), f(x1)
    hist = [x0, x1]
    for _ in range(maxit):
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        hist.append(x2)
        x0, f0, x1, f1 = x1, f1, x2, f(x2)
        if abs(f1) < tol or abs(x1 - x0) < tol:
            break
    return x1, np.array(hist)


def convergence_order(hist, root, lo=1e-13, hi=0.5):
    """Estimate the convergence order p from an iterate history.

    Applies the three-error estimator p ≈ log(e_{n+1}/e_n) / log(e_n/e_{n-1}) (the
    eq-order formula of the theory section) over the clean mid-range: the head has
    not reached the asymptotic regime and the tail is round-off noise, so both are
    excluded by the [lo, hi] error window.

    Parameters
    ----------
    hist : array_like
        The iterate history.
    root : float
        The converged root.
    lo, hi : float, optional
        Error window for the fit.

    Returns
    -------
    float
        The estimated order $p$.
    """
    e = np.abs(np.asarray(hist, dtype=float) - root)
    ps = []
    for n in range(1, len(e) - 1):
        a, b, c = e[n - 1], e[n], e[n + 1]
        if min(a, b, c) > lo and max(a, b, c) < hi and a != b and b != c:
            den = np.log(b / a)
            if den != 0.0:
                ps.append(np.log(c / b) / den)
    return np.array(ps)


def iterations_to_tol(errors, tol=1e-12):
    """Number of steps until the error first drops below a tolerance.

    Parameters
    ----------
    errors : array_like
        The per-step error sequence.
    tol : float, optional
        Target tolerance (default 1e-12).

    Returns
    -------
    int
        The step count to reach ``tol``.
    """
    below = np.where(np.asarray(errors) < tol)[0]
    return int(below[0]) if below.size else len(errors)

## Exercise 1 — Bisection from scratch

Recall the bracketing guarantee behind {eq}`eq-bisection`: if $f(a)f(b)<0$, a
root lies between, and halving the bracket can never lose it. This exercise
implements bisection and runs it on $f(x)=x^2-2$, whose positive root is the
irrational $\sqrt{2}$: a number with no finite decimal, found here purely by
repeated sign tests. {numref}`fig-root-bracket` shows the geometry.

1. Using {eq}`eq-bisection`, take $[a,b]=[1,2]$ (note $f(1)=-1<0$, $f(2)=2>0$)
   and bisect to a tight tolerance with the `bisection` helper, returning the
   midpoint and bracket history.
2. Confirm the result equals $\sqrt{2}$.

In [ ]:
# (solution hidden on the public site)


### Solution

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(root_bis, np.sqrt(2), "bisection converges to √2", rtol=1e-10)

## Exercise 2 — Bisection converges linearly (measured correctly)

The theory, {eq}`eq-bisection`, says the bracket width is $(b-a)_0/2^n$. This
exercise checks that *directly*, and makes a methodological point. The right
quantity to track for bisection is the **bracket width**, which halves cleanly
every step; the midpoint *error* is noisy, because the midpoint can land very
close to the root on one step and then jump away on the next, so error ratios
give a meaningless "order". This is itself worth seeing.

1. From the bracket history of Exercise 1, form the width ratios
   $(b-a)_{n+1}/(b-a)_n$ (an array shift-and-divide) and confirm they equal
   $\tfrac12$.
2. Plot the bracket width against iteration on a log $y$-axis
   (`matplotlib.pyplot.semilogy`): a straight line of slope $-\log 2$, the
   signature of linear convergence ({numref}`fig-root-bisection-rate`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    width_ratios,
    0.5 * np.ones_like(width_ratios),
    "the bracket width halves each step",
    atol=1e-9,
)

## Exercise 3 — Newton's method and quadratic convergence (worked)

Newton's update, {eq}`eq-newton`, steps along the tangent. Near a simple root it
converges *quadratically*, {eq}`eq-order` with $p=2$: the error squares, so
correct digits double per step. This exercise implements Newton on $f(x)=x^2-2$
(with $f'=2x$) and measures the order from the error history.

1. Using {eq}`eq-newton`, iterate from $x_0=2$ with the `newton` helper and
   record the iterates.
2. Using {eq}`eq-order`, estimate the convergence order with the
   `convergence_order` helper, read over the clean mid-range iterates.
3. Plot Newton's error against iteration alongside bisection's on the same log
   axes ({numref}`fig-root-error-compare`): the plunge versus the straight-line
   crawl.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(order_newt, 2.0, "Newton converges quadratically (order ≈ 2)", atol=0.3)

## Exercise 4 — The secant method and the golden ratio

The secant update, {eq}`eq-secant`, is Newton with the derivative replaced by a
finite difference over the last two iterates: no $f'$ required. Its order is
the golden ratio $\varphi=\tfrac{1+\sqrt5}{2}\approx1.618$, between bisection's
$1$ and Newton's $2$.

1. Using {eq}`eq-secant`, iterate on $f(x)=x^2-2$ from $x_0=2,\ x_1=1.5$ with
   the `secant` helper, and estimate the order via {eq}`eq-order` (read over the
   mid-range iterates: the tail is noisy near machine precision).
2. Count iterations to reach a $10^{-12}$ error for all three methods and compare
   them in a bar chart ({numref}`fig-root-iters`): bisection's many steps against
   the handful Newton and the secant need.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    order_sec,
    PHI,
    "the secant method's order is the golden ratio φ",
    atol=0.3,
)

## Exercise 5 — When Newton fails (the cautionary tale)

Newton's speed comes with no guarantee. With the wrong function or start, the
update {eq}`eq-newton` can **cycle** forever or **diverge**. The classic cycle:
on $f(x)=x^3-2x+2$ from $x_0=0$, Newton maps $0\to1\to0\to1\to\cdots$ and never
converges, even though the function has a real root near $-1.77$.

1. Iterate Newton on $f(x)=x^3-2x+2$ from $x_0=0$ and confirm the sequence is
   $0,1,0,1,\dots$.
2. Show the near-zero-derivative overshoot too: start near the critical point
   $f'(x)=0$ at $x=\sqrt{2/3}\approx0.8165$ and watch a single step fling the
   iterate far across the axis.
   {numref}`fig-root-cycling` draws the tangent steps of the cycle.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    np.allclose(cycle[:6], [0, 1, 0, 1, 0, 1]),
    "Newton cycles 0,1,0,1,… and never converges from this start",
    f"first iterates = {cycle[:6]}",
)
validate.check(
    abs(diverge[-1] - (-1.7693)) > 1.0,
    "from a start near f'=0 the near-zero slope flings the iterate far from the root",
    f"last iterate = {diverge[-1]:.3f}",
)

## Exercise 6 — Newton's basins of attraction (worked animation)

Run Newton on the *complex* function $f(z)=z^3-1$, whose three roots are the
cube roots of unity. Every starting point in the plane converges to one of the
three — but the map "start $\to$ which root" has a **fractal** boundary: near
the borders, points arbitrarily close together end up at different roots. This
is the famous Newton fractal: a familiar method producing astonishing
structure.

1. Iterate Newton's update for $f(z)=z^3-1$ on a grid of complex starting
   points (`numpy.meshgrid` over the plane).
2. Colour each start by the root it reaches (`numpy.argmin` of the distances to
   the three roots) and animate the basin map sharpening as the iteration count
   grows ({numref}`fig-root-fractal`).
3. Validate the *mathematics of the data*: starts placed next to each root
   converge to that root, and every limit Newton finds is a genuine root of $f$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
# Starts placed right next to each root must converge to THAT root.
near_starts = roots3 * (1 + 1e-3)
conv = near_starts.copy()
for _ in range(60):
    conv = conv - (conv**3 - 1.0) / (3.0 * conv**2)
validate.close(
    conv, roots3, "starts next to each root converge to that root", rtol=1e-6
)
validate.close(
    conv**3 - 1.0,
    np.zeros(3, dtype=complex),
    "every basin converges to a genuine root of f(z)=z³−1",
    atol=1e-8,
)

## Exercise 7 — The transcendental root that QM needs

Here is that forward promise, made concrete. In
[§6.16](../06-quantum-mechanics/central-potentials-3d.ipynb) the energy levels of the
infinite spherical well (angular momentum $\ell=1$) are the roots of the
transcendental equation $\tan x = x$: no formula gives them; one *must* root-find.
This exercise finds the first nonzero root, $x_\star \approx 4.4934$.

1. Plot $y=\tan x$ and $y=x$ and locate the first few intersections graphically
   ({numref}`fig-root-tanx`), noting the asymptotes of $\tan$ at
   $x=(k+\tfrac12)\pi$: a bracket must not straddle one.
2. The first nonzero root lies between the asymptotes at $\pi/2$ and $3\pi/2$;
   bracket it safely inside $(\pi,\,3\pi/2)$ (e.g. $[4.0,\,4.6]$, clear of the
   asymptote at $3\pi/2\approx4.712$), and solve with the `bisection` helper.
3. Draw the bracket *closing on the root*: a static "ladder" of the successive
   intervals $[a,b]_n$, each row half the width of the one above
   ({numref}`fig-root-tanx-brackets`). (No animation here: the lesson is the
   *nesting* of intervals, which a still ladder shows at a glance.)

The validation checks the *root*, so a ✗ means "check the bracket (did it
cross the $\tan$ asymptote?) or the iteration."

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


Part 3 reconstructs the bracket history $[a,b]_n$ from the bisection run and
draws it as a static ladder: successive rows, each interval half the last,
converging on the root.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    root_tanx,
    4.4934094579,
    "first nonzero root of tan x = x",
    rtol=1e-8,
)

## Exercise 8 — Brent's method: safety and speed (synthesis)

In practice one neither hand-codes bisection nor risks Newton's failures: one calls
**Brent's method**, `scipy.optimize.brentq`. It keeps a bracket (so it inherits
bisection's guarantee, {eq}`eq-bisection`) but interpolates with secant /
inverse-quadratic steps (so it inherits their speed). It is the course default
(for orbital turning points in
[§2.4](../02-classical-mechanics/central-force.ipynb) and the eigenvalue
equations of §6), and it needs only a sign-changing bracket.

1. Re-solve the roots of Exercises 1 and 7 with `scipy.optimize.brentq`.
2. Confirm it lands on the same values the hand-coded solvers found.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(brent_sqrt2, np.sqrt(2), "Brent's method matches √2", rtol=1e-10)
validate.close(
    brent_tanx,
    root_tanx,
    "Brent's method matches the hand-coded tan x = x root",
    rtol=1e-10,
)

## Notebook summary

- Bisection (linear convergence) and Newton's method (**quadratic**, measured order $\approx2$)
  converging to $\sqrt2$; the secant method at the golden-ratio order $\approx1.618$.
- Where Newton fails and its fractal basins of attraction; a transcendental root needed in
  quantum mechanics; and Brent's method as the safe, fast default (`scipy.optimize.brentq`).

## Outlook

- **Multiple roots.** At a repeated root $f'$ also vanishes and Newton's
  convergence degrades from quadratic to *linear*; the modified update
  $x_{n+1}=x_n - m\,f/f'$ (with multiplicity $m$) restores it.
- **Systems of equations.** The multivariate Newton replaces $f'$ with the
  Jacobian matrix and solves a linear system each step: the engine behind
  shooting methods and behind locating the equilibria of
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb).
- **Continuation / homotopy.** Track a root as a parameter varies, using each
  solution as the start for the next: invaluable when a good initial guess is
  otherwise hard to find.
- **Forward links.** [§2.4](../02-classical-mechanics/central-force.ipynb)
  (turning points via `brentq` on $E - V_{\rm eff}$);
  [§6.11](../06-quantum-mechanics/bound-states-1d.ipynb),
  [§6.16](../06-quantum-mechanics/central-potentials-3d.ipynb) (transcendental
  eigenvalues); [§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb) (shooting
  for the hydrogen radial equation).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()